[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/22-regressao-logistica/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/22-regressao-logistica")
    print("Material preparado em:", Path.cwd())


# Regressão logística como modelo linear generalizado

Material de apoio — Aula 22

## Objetivos

Construiremos a regressão logística como GLM Bernoulli, compararemos
funções para mapear o preditor linear em probabilidade, derivaremos a
entropia cruzada e implementaremos o ajuste por gradiente descendente. O
capítulo termina com um exemplo completo usando uma divisão
estratificada de 80% para treino e 20% para teste.

## Como estudar este capítulo

A regressão linear não é adequada quando a resposta assume apenas zero
ou um: a reta pode produzir valores menores que zero ou maiores que um,
e a variabilidade de uma resposta binária não é constante. A regressão
logística preserva a ideia de combinar atributos linearmente, mas
transforma esse escore em uma probabilidade válida.

Há três etapas conceituais. Primeiro, modelamos $Y_i$ por uma Bernoulli.
Depois, construímos o preditor linear $\eta_i=x_i^T\beta$. Por fim,
usamos a função logística para ligar $\eta_i$ à probabilidade $\pi_i$.
Essa decomposição é a ponte para os modelos lineares generalizados.

Durante o estudo, mantenha separadas a estimação, a avaliação e a
decisão. O modelo aprende os coeficientes no treino, o teste avalia seu
comportamento em dados separados e somente depois um limiar transforma
probabilidades em classes. Métricas de discriminação, calibração e
custos dos erros respondem a perguntas diferentes.

## Base de dados de apoio

Usaremos [Medical Insurance
Cost](https://www.kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset).
A resposta será tabagismo; idade, IMC e despesas serão preditores. O
exemplo é didático e descritivo, não uma recomendação de sistema para
inferir comportamento pessoal.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
df = pd.read_csv(Path("../16-correlacao/data/insurance.csv"))
df.head()

In [ ]:
y = (df["smoker"] == "yes").astype(int).to_numpy()
df.groupby("smoker")[["age", "bmi", "charges"]].agg(["count", "mean", "median", "std"]).round(2)

> **Interpretação**
>
> Fumantes representam cerca de 20,5% da base e possuem despesas muito
> maiores. Isso facilita a classificação, mas `charges` pode ser
> posterior ao comportamento e inadequado para uma previsão prospectiva.

## Modelos lineares generalizados

Um modelo linear generalizado mantém o preditor linear da regressão
múltipla, mas permite outras distribuições para a resposta. Ele combina
três componentes:

1.  **componente aleatório:** especifica a distribuição de $Y_i\mid X_i$
    na família exponencial;
2.  **componente sistemático:** reúne os preditores em
    $\eta_i=x_i^T\beta$;
3.  **função de ligação:** relaciona a média condicional ao preditor
    linear por $g(\mu_i)=\eta_i$, em que $\mu_i=E[Y_i\mid X_i]$.

### O que significa família exponencial?

“Família exponencial” descreve uma forma matemática compartilhada por
várias distribuições. Ela não se refere somente à distribuição
exponencial. Uma forma geral é

$$
f(y\mid\theta,\phi)
=\exp\left\{
\frac{y\theta-b(\theta)}{a(\phi)}+c(y,\phi)
\right\},
$$

em que $\theta$ é o parâmetro canônico, $\phi$ controla a dispersão e as
funções $a$, $b$ e $c$ determinam a distribuição específica. Nessa
representação,

$$
E[Y]=b'(\theta),
\qquad
\operatorname{Var}(Y)=a(\phi)b''(\theta).
$$

Portanto, a mesma função $b$ organiza a média e a variância. Normal,
Bernoulli, Binomial, Poisson e Gamma são exemplos importantes dessa
família.

> **Intuição para os GLMs**
>
> A família exponencial oferece uma linguagem comum para respostas
> contínuas, binárias, contagens e valores positivos. Cada distribuição
> mantém seu domínio e sua relação entre média e variância, enquanto o
> modelo preserva o preditor linear, a função de ligação e uma
> verossimilhança tratável. Essa estrutura comum permite desenvolver
> estimação e diagnóstico de maneira unificada.

### A regressão linear normal como GLM

Na regressão linear,

$$
Y_i\mid X_i=x_i\sim N(\mu_i,\sigma^2)
$$

e usamos a ligação identidade:

$$
g(\mu_i)=\mu_i=\eta_i=x_i^T\beta.
$$

A média normal pode assumir qualquer valor real. Por isso, nenhuma
transformação adicional é necessária entre $\mu_i$ e $\eta_i$.

### Da resposta binária ao GLM Bernoulli

$$Y_i\mid X_i=x_i\sim\operatorname{Bernoulli}(\pi_i),$$

$$\eta_i=x_i^T\beta,$$

$$g(\pi_i)=\eta_i.$$

Para uma Bernoulli, $E[Y_i\mid X_i]=\pi_i$. A função de ligação precisa
conectar uma probabilidade restrita a $(0,1)$ ao preditor linear, que
pode assumir qualquer valor real.

> **Interpretação**
>
> A regressão logística preserva a matriz $X$ e o preditor linear
> $X\beta$. A distribuição da resposta, a função de ligação, a
> verossimilhança e a perda mudam em relação à regressão linear normal.

## Testando funções candidatas

Precisamos de $h:\mathbb R\to(0,1)$ com $\pi=h(\eta)$. Uma função linear
viola os limites; um degrau não fornece probabilidade gradual nem
gradiente útil. CDFs contínuas funcionam: Normal produz probit,
logística produz logit e valor extremo produz complementary log-log.

In [ ]:
eta = np.linspace(-6, 6, 400)
candidates = pd.DataFrame({
    "eta": eta,
    "linear": .5+.15*eta,
    "degrau": (eta >= 0).astype(float),
    "arctan": .5+np.arctan(eta)/np.pi,
    "logística": 1/(1+np.exp(-eta)),
})
candidates.head()

> **Interpretação**
>
> Não há uma única função matemática capaz de mapear reais em
> probabilidades. O logit é especialmente conveniente porque lineariza
> os log-odds e fornece razões de odds simples para os coeficientes.

## Odds e interpretação

Odds são $\pi/(1-\pi)$. Como

$$\log(odds)=x^T\beta,$$

uma unidade adicional em $X_j$ multiplica as odds por $e^{\beta_j}$,
mantendo os demais preditores fixos. Isso não é o mesmo que risco
relativo nem implica efeito causal.

## A função logística é a inversa do logit

O logit transforma uma probabilidade em um número real:

$$
\eta=\operatorname{logit}(\pi)
=\log\left(\frac{\pi}{1-\pi}\right).
$$

Para descobrir a transformação inversa, isolamos $\pi$:

$$
\begin{aligned}
\eta&=\log\left(\frac{\pi}{1-\pi}\right)\\
e^\eta&=\frac{\pi}{1-\pi}\\
e^\eta(1-\pi)&=\pi\\
e^\eta-e^\eta\pi&=\pi\\
e^\eta&=\pi+e^\eta\pi=\pi(1+e^\eta)\\
\pi&=\frac{e^\eta}{1+e^\eta}
=\frac{1}{1+e^{-\eta}}
=\sigma(\eta).
\end{aligned}
$$

Portanto,

$$
\operatorname{logit}(\sigma(\eta))=\eta,
\qquad
\sigma(\operatorname{logit}(\pi))=\pi.
$$

In [ ]:
def sigmoid(z):
    return 1/(1+np.exp(-np.clip(z, -30, 30)))

pd.DataFrame({"eta": [-3, 0, 3], "probabilidade": sigmoid(np.array([-3, 0, 3]))})

> **Interpretação**
>
> O logit leva $(0,1)$ a $\mathbb R$. A função logística faz o caminho
> inverso, levando qualquer preditor linear real a uma probabilidade
> válida. O escore zero corresponde a probabilidade 0,5. Escores
> negativos produzem probabilidades abaixo de 0,5 e escores positivos,
> acima.

## Como inserir relações não lineares

O nome “regressão logística linear” refere-se à linearidade nos
coeficientes, não à obrigação de usar apenas os atributos originais. Em
vez de aplicar o modelo diretamente a $x=(x_1,x_2)$, podemos construir
uma transformação $\phi(x)$ e escrever

$$
\eta(x)=\beta^T\phi(x),
\qquad
\pi(x)=\sigma\!\left(\beta^T\phi(x)\right).
$$

Com os atributos originais,

$$
\phi_L(x)=(1,x_1,x_2),
$$

a fronteira de decisão para o limiar 0,5 satisfaz $\pi(x)=0{,}5$, ou,
equivalentemente, $\eta(x)=0$. Portanto,

$$
\beta_0+\beta_1x_1+\beta_2x_2=0,
$$

que descreve uma reta no plano. Para permitir curvatura, podemos
acrescentar termos quadráticos e uma interação:

$$
\phi_Q(x)=(1,x_1,x_2,x_1^2,x_1x_2,x_2^2).
$$

Nesse caso, a condição $\beta^T\phi_Q(x)=0$ descreve uma curva no espaço
original. O modelo ainda é linear em $\beta$: cada coeficiente continua
multiplicando um atributo conhecido. A não linearidade está na
transformação dos atributos.

In [ ]:
rng_curve = np.random.default_rng(20261112)
X_curve = rng_curve.uniform(-2.4, 2.4, size=(360, 2))
true_score = (
    1.0 - .65*X_curve[:, 0]**2
    - .85*X_curve[:, 1] + .25*X_curve[:, 0]
)
y_curve = rng_curve.binomial(1, sigmoid(2.7*true_score))

def design_matrix(points, quadratic=False):
    x1, x2 = points[:, 0], points[:, 1]
    columns = [np.ones(len(points)), x1, x2]
    if quadratic:
        columns.extend([x1**2, x1*x2, x2**2])
    return np.column_stack(columns)

def fit_logistic(design, response, learning_rate, steps=7000):
    beta = np.zeros(design.shape[1])
    for _ in range(steps):
        probability = sigmoid(design @ beta)
        gradient = design.T @ (probability - response) / len(response)
        beta = beta - learning_rate*gradient
    return beta

beta_linear = fit_logistic(design_matrix(X_curve), y_curve, .12)
beta_quadratic = fit_logistic(design_matrix(X_curve, True), y_curve, .025)

grid_axis = np.linspace(-2.5, 2.5, 240)
grid_x1, grid_x2 = np.meshgrid(grid_axis, grid_axis)
grid = np.column_stack([grid_x1.ravel(), grid_x2.ravel()])
grid_probabilities = [
    sigmoid(design_matrix(grid) @ beta_linear).reshape(grid_x1.shape),
    sigmoid(design_matrix(grid, True) @ beta_quadratic).reshape(grid_x1.shape),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True, sharey=True)
titles = ["Somente termos lineares", "Com termos quadráticos"]
for ax, probability_grid, title in zip(axes, grid_probabilities, titles):
    ax.contourf(
        grid_x1, grid_x2, probability_grid,
        levels=[0, .5, 1], colors=["#f7c9b2", "#b9dce5"], alpha=.45
    )
    ax.contour(grid_x1, grid_x2, probability_grid, levels=[.5],
               colors=["#25364a"], linewidths=2.5)
    ax.scatter(X_curve[y_curve == 0, 0], X_curve[y_curve == 0, 1],
               c="#d95f02", marker="x", s=22, label="classe 0")
    ax.scatter(X_curve[y_curve == 1, 0], X_curve[y_curve == 1, 1],
               facecolors="white", edgecolors="#1f6f8b", s=22, label="classe 1")
    ax.set(title=title, xlabel="$x_1$", ylabel="$x_2$")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

> **Interpretação**
>
> Os mesmos pontos são mostrados nos dois gráficos. O modelo com apenas
> $x_1$ e $x_2$ só consegue traçar uma reta. Ao incluir $x_1^2$,
> $x_1x_2$ e $x_2^2$, a fronteira pode acompanhar a estrutura curva.
> Outras transformações, como $\log(x)$, raízes, interações e *splines*,
> seguem a mesma ideia. A flexibilidade adicional deve ser avaliada no
> conjunto de teste, pois muitos termos também podem produzir
> sobreajuste.

## Verossimilhança e entropia cruzada

$$L(\beta)=\prod_i\pi_i^{y_i}(1-\pi_i)^{1-y_i}.$$

A log-verossimilhança negativa média é

$$
J(\beta)=-\frac1n\sum_i[y_i\log\pi_i+(1-y_i)\log(1-\pi_i)].
$$

Previsões confiantes e erradas recebem perda muito grande.

## Derivada da função logística

Antes de derivar a perda, precisamos entender como a probabilidade muda
quando o preditor linear muda. Escrevendo

$$
\pi=\sigma(\eta)=\frac{1}{1+e^{-\eta}}=(1+e^{-\eta})^{-1},
$$

obtemos, pela regra da cadeia,

$$
\begin{aligned}
\frac{d\pi}{d\eta}
&=-(1+e^{-\eta})^{-2}\frac{d}{d\eta}(1+e^{-\eta})\\
&=-(1+e^{-\eta})^{-2}(-e^{-\eta})\\
&=\frac{e^{-\eta}}{(1+e^{-\eta})^2}.
\end{aligned}
$$

Agora separamos o resultado em dois fatores:

$$
\frac{e^{-\eta}}{(1+e^{-\eta})^2}
=\underbrace{\frac{1}{1+e^{-\eta}}}_{\pi}
\underbrace{\frac{e^{-\eta}}{1+e^{-\eta}}}_{1-\pi}
=\pi(1-\pi).
$$

O segundo fator é $1-\pi$ porque

$$
1-\frac{1}{1+e^{-\eta}}
=\frac{e^{-\eta}}{1+e^{-\eta}}.
$$

> **Interpretação**
>
> A derivada atinge seu maior valor quando $\pi=0{,}5$. Perto de 0 ou 1,
> a curva logística fica mais plana: uma mesma mudança em $\eta$ provoca
> uma mudança menor na probabilidade.

## Gradiente da log-loss

Para uma observação, a perda é

$$
j_i(\beta)
=-\left[y_i\log\pi_i+(1-y_i)\log(1-\pi_i)\right],
$$

em que $\pi_i=\sigma(\eta_i)$ e $\eta_i=x_i^T\beta$. Para calcular a
derivada em relação ao coeficiente $\beta_j$, seguimos a cadeia

$$
\beta_j\longmapsto\eta_i\longmapsto\pi_i\longmapsto j_i.
$$

Portanto, a regra da cadeia decompõe a derivada em três fatores:

$$
\frac{\partial j_i}{\partial\beta_j}
=
\frac{\partial j_i}{\partial\pi_i}
\frac{\partial\pi_i}{\partial\eta_i}
\frac{\partial\eta_i}{\partial\beta_j}.
$$

Derivamos primeiro a perda em relação à probabilidade, depois a
probabilidade em relação ao preditor linear e, por fim, o preditor
linear em relação ao coeficiente $\beta_j$.

O fator $x_{ij}$ aparece ao derivarmos o preditor linear. Para a
observação $i$,

$$
\eta_i=x_i^T\beta
=\beta_0+\beta_1x_{i1}+\cdots+\beta_jx_{ij}+\cdots+\beta_px_{ip}.
$$

Ao derivar em relação a $\beta_j$, todos os outros termos permanecem
constantes. O único termo que depende de $\beta_j$ é $\beta_jx_{ij}$.
Logo,

$$
\frac{\partial\eta_i}{\partial\beta_j}
=\frac{\partial}{\partial\beta_j}(\beta_jx_{ij})
=x_{ij}.
$$

Para o primeiro fator, usamos $d\log(\pi_i)/d\pi_i=1/\pi_i$ e
$d\log(1-\pi_i)/d\pi_i=-1/(1-\pi_i)$:

$$
\begin{aligned}
\frac{\partial j_i}{\partial\pi_i}
&=-\left[
\frac{y_i}{\pi_i}-\frac{1-y_i}{1-\pi_i}
\right]\\
&=-\frac{y_i}{\pi_i}+\frac{1-y_i}{1-\pi_i}.
\end{aligned}
$$

Os outros dois fatores são

$$
\frac{\partial\pi_i}{\partial\eta_i}=\pi_i(1-\pi_i),
\qquad
\frac{\partial\eta_i}{\partial\beta_j}=x_{ij}.
$$

Multiplicando-os pela regra da cadeia,

$$
\begin{aligned}
\frac{\partial j_i}{\partial\beta_j}
&=\left[-\frac{y_i}{\pi_i}+\frac{1-y_i}{1-\pi_i}\right]
\pi_i(1-\pi_i)x_{ij}\\
&=\left[-y_i(1-\pi_i)+(1-y_i)\pi_i\right]x_{ij}\\
&=(\pi_i-y_i)x_{ij}.
\end{aligned}
$$

Como $J(\beta)=n^{-1}\sum_i j_i(\beta)$, o componente $j$ do gradiente é

$$
\frac{\partial J}{\partial\beta_j}
=\frac1n\sum_{i=1}^n x_{ij}(\pi_i-y_i).
$$

Reunindo todos os componentes em um vetor,

$$
\nabla J(\beta)=\frac1nX^T(\pi-y).
$$

Cada linha de $X$ representa uma pessoa e cada coluna representa o
intercepto ou um atributo. O vetor $\pi-y$ contém as diferenças entre
probabilidade prevista e resposta observada. O produto por $X^T$
acumula, para cada coeficiente, as contribuições de todas as
observações.

> **Interpretação**
>
> Se $\pi_i>y_i$, a observação empurra o coeficiente na direção que
> reduz a probabilidade prevista. Se $\pi_i<y_i$, ela produz o movimento
> contrário. O fator $x_{ij}$ determina quanto o atributo $j$ participa
> dessa correção.

## Extensão para mais de duas classes

A regressão logística construída até aqui resolve um problema binário.
Quando a resposta possui $K>2$ classes, podemos reutilizar esse mesmo
classificador por meio de duas decomposições.

### One-vs-all: uma classe contra todas as demais

Também chamada de *one-vs-rest*, essa estratégia ajusta $K$ modelos.
Para cada classe $k$, criamos temporariamente a resposta binária

$$
Y_i^{(k)}=
\begin{cases}
1, & \text{se }Y_i=k,\\
0, & \text{se }Y_i\neq k.
\end{cases}
$$

O modelo $k$ aprende a separar a classe $k$ da reunião de todas as
outras. Para uma nova observação, calculamos os $K$ escores ou
probabilidades e escolhemos a classe associada ao maior valor. Com
quatro classes, por exemplo, ajustamos quatro modelos.

Essa abordagem usa poucos modelos, mas o grupo “todas as outras classes”
pode reunir observações muito diferentes entre si. Além disso, como as
probabilidades foram produzidas por modelos independentes, elas não
precisam somar exatamente 1 sem uma etapa adicional de calibração.

### One-vs-one: comparação entre pares

Nessa estratégia, ajustamos um classificador para cada par não ordenado
de classes. O número total de modelos é

$$
\binom{K}{2}=\frac{K(K-1)}{2}.
$$

Cada ajuste utiliza somente as observações pertencentes às duas classes
do par. Na previsão, cada classificador dá um voto e escolhemos a classe
com mais votos. Com quatro classes, são necessários $\binom{4}{2}=6$
modelos.

Os problemas binários podem ficar mais simples porque cada modelo
compara apenas duas classes. Em contrapartida, o número de ajustes
cresce quadraticamente com $K$ e pode haver empates, que precisam de uma
regra de desempate.

| Critério | One-vs-all | One-vs-one |
|------------------------|------------------------|------------------------|
| Número de modelos | $K$ | $K(K-1)/2$ |
| Custo de treinamento | geralmente menor e mais rápido | tende a crescer mais rapidamente com $K$; cada modelo, porém, usa apenas duas classes |
| Decisão final | maior escore ou probabilidade | maioria dos votos |
| Desequilíbrio entre classes | mais exposto quando a união das demais classes domina a classe positiva | em geral menos exposto por comparar apenas duas classes, mas o par ainda pode ser desbalanceado |

A vantagem do one-vs-one diante do desbalanceamento é relativa, não uma
garantia. Se uma das duas classes do par tiver muito mais observações,
ainda pode ser necessário usar pesos de classe, subamostragem ou
sobreamostragem. Do mesmo modo, uma implementação one-vs-all com pesos
adequados pode lidar bem com classes de tamanhos diferentes.

> **Interpretação**
>
> As duas estratégias transformam um problema multiclasse em vários
> problemas binários. A regressão logística multinomial com *softmax* é
> uma alternativa direta: ajusta as classes conjuntamente e produz
> probabilidades que somam 1.

## Exemplo completo: divisão 80% treino e 20% teste

Usaremos uma única divisão estratificada. Isso preserva aproximadamente
a proporção de fumantes nos dois conjuntos. Não faremos validação
cruzada nesta aula.

In [ ]:
features = ["age", "bmi", "charges"]
X_raw = df[features].to_numpy(float)

generator = np.random.default_rng(20261111)
test_parts = []
for label in [0, 1]:
    label_ids = np.flatnonzero(y == label)
    generator.shuffle(label_ids)
    test_parts.append(label_ids[:round(.20 * len(label_ids))])

test_ids = np.concatenate(test_parts)
generator.shuffle(test_ids)
train_ids = np.setdiff1d(np.arange(len(y)), test_ids)

y_train = y[train_ids]
y_test = y[test_ids]

pd.DataFrame({
    "conjunto": ["treino", "teste"],
    "n": [len(train_ids), len(test_ids)],
    "fumantes": [y_train.sum(), y_test.sum()],
    "proporção_fumantes": [y_train.mean(), y_test.mean()],
}).round(3)

> **Interpretação**
>
> O treino contém 1.070 pessoas e o teste, 268. As proporções de
> fumantes ficam próximas porque a divisão ocorreu separadamente dentro
> de cada classe. O teste será usado apenas depois que a preparação e o
> ajuste terminarem.

## Padronização estimada somente no treino

Média e desvio padrão também são parâmetros aprendidos. Por isso,
calculamos essas quantidades no treino e reutilizamos exatamente os
mesmos valores no teste.

In [ ]:
train_mean = X_raw[train_ids].mean(axis=0)
train_sd = X_raw[train_ids].std(axis=0)

Z_train = (X_raw[train_ids] - train_mean) / train_sd
Z_test = (X_raw[test_ids] - train_mean) / train_sd

X_train = np.column_stack([np.ones(len(train_ids)), Z_train])
X_test = np.column_stack([np.ones(len(test_ids)), Z_test])

pd.DataFrame({
    "atributo": features,
    "média_treino": train_mean,
    "desvio_treino": train_sd,
}).round(3)

> **Evite vazamento de informação**
>
> Calcular média e desvio padrão na base completa deixaria o teste
> influenciar a preparação. A contaminação seria pequena neste exemplo,
> mas o procedimento seria conceitualmente incorreto e pode causar erros
> maiores em outras bases.

## Ajuste por gradiente descendente no treino

Escrevemos a perda e o gradiente como funções dos dados recebidos.
Assim, o código deixa explícito que o ajuste usa apenas `X_train` e
`y_train`.

A atualização usa

$$
\beta_{t+1}
=\beta_t-\kappa\nabla J(\beta_t)
=\beta_t-\kappa\frac{1}{n}X^T(\pi_t-y),
\qquad
\pi_t=\sigma(X\beta_t),
$$

em que $\kappa>0$ é a taxa de aprendizado. O gradiente aponta para o
crescimento mais rápido da log-loss; a subtração é o que transforma essa
direção em uma etapa de descida. Nesta aula, $\eta_i=x_i^T\beta$ fica
reservado ao preditor linear do GLM.

In [ ]:
def loss_grad(beta, X, y):
    probability = sigmoid(X @ beta)
    loss = -np.mean(
        y * np.log(probability + 1e-12)
        + (1 - y) * np.log(1 - probability + 1e-12)
    )
    gradient = X.T @ (probability - y) / len(y)
    return loss, gradient, probability

beta = np.zeros(X_train.shape[1])
history = []

for iteration in range(3000):
    loss, gradient, p_train = loss_grad(beta, X_train, y_train)
    history.append(loss)
    beta = beta - .15 * gradient

coefficient_table = pd.DataFrame({
    "termo": ["intercepto"] + features,
    "beta": beta,
    "razão_de_odds": np.exp(beta),
})
coefficient_table.round(4)

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(history)
plt.xlabel("iteração")
plt.ylabel("entropia cruzada no treino")
plt.show()

> **Interpretação**
>
> O coeficiente de `charges` é positivo e muito maior em magnitude. Como
> os atributos foram padronizados, sua razão de odds compara pessoas
> separadas por um desvio padrão de despesas, mantendo idade e IMC
> fixos. Esses valores descrevem associação, e não efeito causal.

## Probabilidades no conjunto de teste

Somente agora usamos `X_test`. O modelo transforma cada linha em uma
probabilidade sem consultar a classe verdadeira durante a previsão.

In [ ]:
_, _, p_test = loss_grad(beta, X_test, y_test)

test_probabilities = pd.DataFrame({
    "probabilidade": p_test,
    "classe": np.where(y_test == 1, "fumante", "não fumante"),
})

plt.figure(figsize=(9, 4.5))
sns.histplot(
    data=test_probabilities,
    x="probabilidade",
    hue="classe",
    bins=18,
    element="step",
    stat="density",
    common_norm=False,
)
plt.axvline(.5, color="black", linestyle="--", label="limiar 0,5")
plt.xlabel("probabilidade prevista no teste")
plt.show()

> **Interpretação**
>
> As distribuições ficam bem separadas, mas ainda existe sobreposição. O
> teste é mais honesto que a avaliação no treino porque contém pessoas
> que não participaram da estimação dos coeficientes.

## Classes, matriz de confusão e métricas

Com o limiar $c=0{,}5$, calculamos as quatro contagens da matriz de
confusão e as métricas derivadas delas.

In [ ]:
def confusion_counts(y, prediction):
    tp = int(np.sum(prediction & (y == 1)))
    fn = int(np.sum(~prediction & (y == 1)))
    tn = int(np.sum(~prediction & (y == 0)))
    fp = int(np.sum(prediction & (y == 0)))
    return tn, fp, fn, tp

def classification_metrics(y, probability, threshold=.5):
    prediction = probability >= threshold
    tn, fp, fn, tp = confusion_counts(y, prediction)
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    precision = tp / (tp + fp) if tp + fp else np.nan
    accuracy = (tp + tn) / len(y)
    f1 = 2 * precision * sensitivity / (precision + sensitivity)
    return {
        "limiar": threshold,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "acurácia": accuracy,
        "sensibilidade": sensitivity,
        "especificidade": specificity,
        "precisão": precision,
        "F1": f1,
    }

test_metrics = classification_metrics(y_test, p_test, threshold=.5)
pd.Series(test_metrics).round(4)

> **Interpretação**
>
> No teste, o modelo produz 210 verdadeiros negativos, 47 verdadeiros
> positivos, 3 falsos positivos e 8 falsos negativos. A acurácia de
> 95,9% não mostra sozinha que a sensibilidade, também chamada
> **revocação**, é menor, 85,5%: o modelo deixa de identificar parte dos
> fumantes.

## O limiar redistribui os tipos de erro

In [ ]:
threshold_results = pd.DataFrame([
    classification_metrics(y_test, p_test, threshold)
    for threshold in [.20, .35, .50, .65, .80]
])
threshold_results.set_index("limiar")[[
    "sensibilidade", "especificidade", "precisão", "acurácia"
]].round(3)

In [ ]:
threshold_grid = np.linspace(.05, .95, 37)
threshold_curve = pd.DataFrame([
    classification_metrics(y_test, p_test, threshold)
    for threshold in threshold_grid
])

plt.figure(figsize=(9, 4.5))
for metric in ["sensibilidade", "especificidade", "precisão"]:
    plt.plot(threshold_curve["limiar"], threshold_curve[metric], label=metric)
plt.axvline(.5, color="black", linestyle="--")
plt.xlabel("limiar")
plt.ylabel("proporção")
plt.legend()
plt.show()

> **Interpretação**
>
> Reduzir o limiar recupera mais casos da classe 1 e aumenta a
> sensibilidade, mas também produz mais falsos positivos e reduz a
> especificidade. Aumentar o limiar faz o movimento contrário: exige
> maior probabilidade para prever a classe 1, reduz falsos positivos e
> aumenta falsos negativos. A precisão não precisa variar
> monotonicamente em uma amostra finita, porque cada mudança de limiar
> pode mover poucas observações de uma vez. O limiar 0,5 é apenas uma
> referência; a escolha deve refletir as consequências dos dois tipos de
> erro.

## Calibração no teste

Para formar seis grupos de tamanhos semelhantes mesmo quando há
probabilidades muito próximas, usamos o posto das previsões somente na
definição das faixas. As médias mostradas continuam sendo as
probabilidades originais.

In [ ]:
calibration_data = pd.DataFrame({"p": p_test, "y": y_test})
calibration_data["faixa"] = pd.qcut(
    calibration_data["p"].rank(method="first"), 6
)

calibration = calibration_data.groupby("faixa", observed=True).agg(
    probabilidade_média=("p", "mean"),
    frequência_observada=("y", "mean"),
    n=("y", "size"),
).reset_index(drop=True)
calibration["grupo"] = [f"G{i}" for i in range(1, len(calibration)+1)]

brier = np.mean((p_test - y_test) ** 2)
print("Brier:", round(brier, 4))
calibration.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [1.35, 1]})
axes[0].plot([0, 1], [0, 1], "--", color="black")
axes[0].scatter(
    calibration["probabilidade_média"],
    calibration["frequência_observada"],
    s=calibration["n"] * 5,
)
for row in calibration.iloc[4:].itertuples():
    axes[0].annotate(row.grupo, (row.probabilidade_média, row.frequência_observada))
axes[0].annotate(
    "G1–G4 sobrepostos",
    xy=(calibration.loc[:3, "probabilidade_média"].mean(), 0),
    xytext=(.08, .13), arrowprops={"arrowstyle": "->"},
)
axes[0].set(
    xlim=(0, 1), ylim=(0, 1), xlabel="probabilidade média prevista",
    ylabel="proporção observada de fumantes", title="Visão completa",
)

low_groups = calibration.iloc[:4]
axes[1].plot([0, .025], [0, .025], "--", color="black")
axes[1].scatter(
    low_groups["probabilidade_média"], low_groups["frequência_observada"],
    s=low_groups["n"] * 5,
)
for row in low_groups.itertuples():
    axes[1].annotate(row.grupo, (row.probabilidade_média, row.frequência_observada),
                     xytext=(0, 12), textcoords="offset points", ha="center")
axes[1].set(
    xlim=(-.001, .022), ylim=(-.002, .03), xlabel="probabilidade média prevista",
    ylabel="proporção observada", title="Zoom: grupos de baixa probabilidade",
)
plt.tight_layout()
plt.show()

> **Interpretação**
>
> O escore de Brier é aproximadamente 0,025 e vale zero para previsões
> perfeitas. G1–G4 possuem probabilidades médias entre aproximadamente
> 0,0003 e 0,014 e nenhum fumante observado; por isso aparecem
> sobrepostos na escala completa. G5 tem probabilidade média próxima de
> 0,22 e frequência observada de 0,30, enquanto G6 tem valores próximos
> de 0,91 e 0,93. Os grupos ficam razoavelmente próximos da diagonal,
> mas 268 casos ainda formam uma amostra pequena para conclusões fortes.
> Discriminação e calibração continuam sendo propriedades diferentes.

## Efeito marginal

$$\frac{\partial\pi}{\partial x_j}=\beta_j\pi(1-\pi).$$

Portanto, o efeito na escala de probabilidade depende da própria
probabilidade. Ele é maior perto de 0,5 e menor nos extremos.

> **Interpretação**
>
> Uma razão de odds é constante no modelo, mas a mudança em pontos
> percentuais não é. Por isso, gráficos de probabilidades previstas
> costumam comunicar melhor o efeito substantivo.

## Separação como limitação

Separação perfeita ocorre quando alguma combinação dos preditores
distingue integralmente as duas classes. Nesse caso, a verossimilhança
pode continuar aumentando quando a magnitude dos coeficientes cresce, e
o MLE finito pode não existir.

Coeficientes muito grandes e falhas de convergência são sinais
importantes. O tratamento desse problema será estudado na aula seguinte,
dedicada à regularização.

## Como ler um coeficiente

O coeficiente atua primeiro sobre o logaritmo das odds. Exponenciá-lo
produz uma razão de odds. Se $e^{\beta_j}=1{,}5$, uma unidade adicional
no atributo está associada a odds 50% maiores, mantendo os outros
atributos constantes.

Isso não quer dizer que a probabilidade aumente 50%. A mesma razão de
odds pode representar uma pequena mudança quando a probabilidade inicial
é baixa e uma mudança diferente perto de 0,5. Para comunicação, é útil
mostrar probabilidades previstas para alguns perfis concretos.

> **Momento da previsão**
>
> Um atributo só pode entrar se estiver disponível quando a previsão
> será feita. Neste exemplo, `charges` ajuda a identificar tabagismo,
> mas pode ser posterior ao comportamento e inadequado para uma
> aplicação prospectiva. O exercício ensina o fluxo computacional e não
> propõe um sistema para inferir características pessoais.

## Questões de revisão

1.  Quais são os três componentes do GLM logístico?
2.  Por que linear e degrau são escolhas ruins para $h(\eta)$?
3.  Derive a inversa do logit.
4.  Interprete $e^{\beta_j}$ e diferencie OR de RR.
5.  Explique a forma do gradiente $X^T(\pi-y)/n$ e por que ele é
    subtraído na atualização.
6.  Diferencie probabilidade, limiar, classe, discriminação e
    calibração.

## Bibliografia

-   James et al., *An Introduction to Statistical Learning*, capítulo 4.
-   Gelman, Hill e Vehtari, *Regression and Other Stories*.
-   Fox, *Applied Regression Analysis and Generalized Linear Models*.
-   McCullagh e Nelder, *Generalized Linear Models*.
-   Data 100, classificação e regressão logística.